# Семинар 8. Интеграция AI-моделей в Power BI и создание итогового дашборда

**Цель семинара:** Собрать воедино все артефакты, созданные за семестр, и разработать финальный интерактивный дашборд руководителя. Вам не нужно быть профессиональным разработчиком на Python или экспертом в Power BI — следуйте пошаговым инструкциям. Вы импортируете очищенные данные, интегрируете модель машинного обучения напрямую в Power Query и построите симулятор бизнес-решений (What-If), который докажет экономическую эффективность вашего AI-решения.

Этот дашборд — кульминация вашей курсовой работы. Он рассказывает связную историю: от выявления проблем в сырых данных (Семинар 1) до поведенческой сегментации (Семинар 2), NLP-анализа (Семинар 4), машинного обучения (Семинар 5-6) и финансового прогнозирования (Семинар 7).

---

## ⚙️ Шаг 1. Автоматическая настройка связки Power BI и Python

Power BI "из коробки" не знает, где лежат библиотеки вашего проекта. Чтобы избежать ошибок совместимости и системных сбоев, мы автоматизировали настройку среды.

**Инструкция:**
1. Закройте Power BI Desktop (если он открыт).
2. Откройте **PowerShell** в корневой папке вашего проекта.
3. Выполните готовый скрипт автоматической настройки:
```powershell
Set-ExecutionPolicy -ExecutionPolicy Unrestricted -Scope Process; .\.vscode\setup-powerbi-python.ps1
```
или
```powershell
pwsh -ExecutionPolicy Unrestricted -File .\.vscode\setup-powerbi-python.ps1
```

4. Откройте Power BI Desktop. Перейдите в **Файл -> Параметры и настройки -> Параметры -> Скрипты Python**.
5. В выпадающем списке «Обнаруженные домашние каталоги Python» выберите вашу виртуальную среду (она будет заканчиваться на `-maiba`). Нажмите **ОК**.

---

## 📥 Шаг 2. Загрузка аналитических витрин

Нам понадобятся два главных файла из предыдущих семинаров.

**Инструкция:**

1. В Power BI нажмите **Получить данные -> Текстовый/CSV-файл**.
2. Загрузите `abt_result.csv` (ваша главная витрина из Семинара 5). Нажмите **Преобразовать данные** (откроется редактор Power Query).
3. Таким же образом загрузите файл `forecast.csv` (прогноз временных рядов из Семинара 7).

---

## 🧠 Шаг 3. Интеграция Pickle-модели в Power Query (AI Scoring)

Прямо внутри Power Query мы прогоним нашу витрину через ML-модель из Семинара 6, чтобы каждый клиент получил вероятность наступления целевого события (Отток, Дефолт, Конверсия и т.д.).

**Инструкция:**

1. В Power Query выберите таблицу `abt_result`.
2. На верхней панели перейдите во вкладку **Преобразование** и нажмите **Запустить скрипт Python**.
3. В открывшееся окно скопируйте следующий готовый код. **Обязательно замените пути** на свои реальные абсолютные пути к проекту!


In [ ]:
import site
import sys
import joblib
import pandas as pd

# 1. Безопасное подключение библиотек виртуального окружения

# ВНИМАНИЕ: Замените на абсолютный путь к папке site-packages вашего .venv

venv_packages = r"C:\Users\Student\Projects\course_project.venv\Lib\site-packages"
site.addsitedir(venv_packages)

# 2. Загрузка обученной модели из Семинара 6

# ВНИМАНИЕ: Замените на абсолютный путь к вашему model.pkl

model_path = r"C:\Users\Student\Projects\course_project\data\seminar_6_automl_shap\models\model.pkl"
model = joblib.load(model_path)

# Power BI автоматически передает текущую таблицу в переменную 'dataset'

X = dataset.copy()

# 3. Изоляция целевых и сырых фичей от модели

# Алгоритм должен получать только те столбцы, на которых обучался

target_candidates = ["Target_ID", "Target_Flag", "Churn", "Exited", "Default", "Converted", "Cancelled", "Lapsed", "Retained", "Abandoned", "Dropped", "NoShow", "Fraud", "Stale", "Renewed", "Churned"]
cols_to_drop = [c for c in dataset.columns if c in target_candidates or str(c).endswith('_Raw')]
X_features = X.drop(columns=cols_to_drop, errors="ignore")

# 4. Получение вероятности (Risk Score) и метки класса

if hasattr(model, "predict_proba"):
    dataset["Risk_Score"] = model.predict_proba(X_features)[:, 1]
else:
    dataset["Risk_Score"] = model.predict(X_features)


4. Нажмите **ОК**.
5. В появившемся окне нажмите на слово **Table** (зеленая гиперссылка) напротив строки `dataset`. Таблица развернется с новой колонкой: `Risk_Score`.
6. Нажмите **Закрыть и применить** на главной вкладке Power Query.

---

## 🎛 Шаг 4. Построение Симулятора (What-If) и 3 Синергетические Меры (DAX)

Наша задача — не просто показать риск, а дать бизнесу инструмент управления. Мы создадим слайдер, с помощью которого руководитель сможет менять порог чувствительности алгоритма и сразу видеть экономический эффект.

### 1. Создание параметра What-If (Слайдера):

Перейдите во вкладку **Моделирование -> Создать параметр -> Числовой диапазон**.

* **Имя**: `Порог_Риска`
* **Тип данных**: Десятичное число
* **Минимум**: `0.0` | **Максимум**: `1.0` | **Шаг**: `0.05` | **По умолчанию**: `0.5`
* Поставьте галочку **«Добавить срез на эту страницу»**.

### 2. Создание 3 Бизнес-Метрик (Measures):

Нажмите правой кнопкой мыши по таблице `abt_result` -> **Создать меру**. Создайте по очереди 3 метрики:

> *Внимание: замените `[Monetary_Raw]` на реальное название вашей финансовой колонки (например, `TotalCharges_Raw`, `Balance_Raw`, `Price_Raw` в зависимости от вашего варианта).*

**Мера 1: Клиенты в зоне риска** (Количество клиентов, чей скор превышает выбранный порог)

```dax
Клиенты_В_Риске = 
VAR Threshold = [Значение Порог_Риска]
RETURN
CALCULATE(
    COUNTROWS('abt_result'),
    'abt_result'[Risk_Score] >= Threshold
)

```

**Мера 2: Капитал под угрозой** (Сумма потенциально потерянных денег)

```dax
Капитал_В_Риске = 
VAR Threshold = [Значение Порог_Риска]
RETURN
CALCULATE(
    SUM('abt_result'[Monetary_Raw]),
    'abt_result'[Risk_Score] >= Threshold
)

```

**Мера 3: Чистая спасенная прибыль (ROI)** (Синергия ML и экономики: предполагаем стоимость удержания $10 на клиента, а успех удержания = 20%).

```dax
Чистая_Прибыль_Кампании = 
VAR Cost_Per_Action = 10
VAR Campaign_Cost = [Клиенты_В_Риске] * Cost_Per_Action
VAR Saved_Revenue = [Капитал_В_Риске] * 0.20
RETURN
Saved_Revenue - Campaign_Cost

```

Разместите эти три меры в виде визуалов **«Карточка» (Card)** на вашем дашборде. Теперь при перемещении ползунка бизнес-кейс пересчитывается в реальном времени!

---

## 🎨 Шаг 5. Композиция итогового дашборда (Архитектура визуализации)

Ваш дашборд должен быть разделен на смысловые блоки, иллюстрирующие результаты каждого этапа работы за семестр:

### Блок 1. Проблематика, профили и клиентские сегменты (Семинары 1-4)

* **Профили и очистка (Семинар 1 & 3):** Гистограмма или кольцевая диаграмма распределения клиентов по категориям лояльности `Loyalty_Tier` или `tenure`.
* **RFM Сегментация (Семинар 2):** Точечная диаграмма (Scatter plot). Ось X: `Frequency`, Ось Y: `Monetary_Raw`, Легенда (Цвет): `Cluster_ID`. Показывает ценность выделенных поведенческих сегментов.
* **Голос клиента / Sentiment (Семинар 4):** Датчик (Gauge) или карточка со средним значением `Mean_Sentiment`. Иллюстрирует удовлетворенность клиентов по данным NLP-анализа отзывов.

### Блок 2. Машинное обучение, What-If и Прогнозирование (Семинары 5-7)

* **AI Симулятор (Семинар 5 & 6):** Срез-слайдер `Порог_Риска` и 3 созданные DAX-карточки (`Клиенты_В_Риске`, `Капитал_В_Риске`, `Чистая_Прибыль_Кампании`).
* **Операционный список удержания:** Таблица (Table), содержащая ID клиента, `Risk_Score`, `Cluster_ID` и финансовую ценность (`Monetary_Raw`). Отфильтруйте её визуал, чтобы показывать только клиентов выше порога риска.
* **Временной прогноз (Семинар 7):** График с областями или линейный график из таблицы `forecast.csv`. Ось X: Дата, Ось Y: `yhat` (Прогноз), Доверительный коридор: `Optimistic_Forecast` и `Pessimistic_Forecast`.

---

## 📋 Требования к успешной сдаче (Критерии оценки)

Для успешной защиты курсового проекта ваш дашборд должен соответствовать следующим критериям:

1. **Техническая интеграция:** Python-скрипт в Power Query отрабатывает без ошибок, модель `.pkl` успешно загружается локально, столбцы с вероятностями (`Risk_Score`) генерируются автоматически.
2. **Интерактивность:** Параметр What-If (`Порог_Риска`) работает. При изменении ползунка три главные карточки (Клиенты, Капитал, Прибыль) пересчитываются моментально.
3. **Корректность экономика:** В DAX-мерах используются ненормализованные финансовые показатели (суффикс `_Raw`), чтобы отображались реальные деньги, а не нормированные дроби из StandardScaler.
4. **Полнота покрытия:** На листе присутствуют графики, визуализирующие результаты всех этапов: очистку, кластеризацию, NLP-анализ тональности, ML-моделирование и прогнозирование Prophet.
5. **Бизнес-оформление:** Отсутствуют дефолтные технические названия (например, `Sum of MonthlyCharges_Raw`). Все заголовки графиков и названия осей переведены на понятный бизнес-язык.
